# Part B: Matrices

**Final Project: Philippine Peso Depreciation Early-Warning Model: Predicting the Next-Month PHP/USD Exchange Rate Using Linear Regression and PCA**

Your entire dataset can be written as a single matrix. This is the object every later step operates on.

---
**Student name:** Alfajaro, Ghan Vincent | Arile, Charles Jullianne | Asuar, Jhoe Maven | Mondragon, Leneth | Ulitin, Justin |      **Date:** November 16, 2026

---

**What to do:** Represent your dataset as a design matrix X ∈ ℝⁿˣᵖ. Compute the product XᵀX.

In [1]:
# Run this first: recall variables saved from the previous part's notebook.
# If the store is empty (Part A was not run in this session), the proposal's
# five sample rows are loaded instead so this notebook still runs end to end.
%store -r X_raw y feature_names

try:
    _ = (X_raw, y, feature_names)
    print("Loaded X_raw, y, feature_names from Part A.")
except NameError:
    print("Store is empty: loading the full dataset from peso_usd_part_a_dataset.csv (saved by Part A)...")
    df = pd.read_csv("peso_usd_part_a_dataset.csv", index_col="FeatureMonth")
    feature_names = ["CurrentRate", "Trailing3MonthAverage", "YearOverYearChange"]
    X_raw = df[feature_names].to_numpy(dtype=float)
    y = df["NextMonthRate"].to_numpy(dtype=float)
    print("Loaded", X_raw.shape, "from the CSV.")

Loaded X_raw, y, feature_names from Part A.


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)

In [3]:
# Build the design matrix X (rows = months, columns = the 3 features).
# The target (NextMonthRate) stays in y and is NOT a column of X.
X = np.asarray(X_raw, dtype=float)
y = np.asarray(y, dtype=float)

n, p = X.shape
print("X shape:", X.shape)
print("n (rows, months) =", n, "| p (columns, features) =", p)
print("y shape:", y.shape)
print("Any missing values in X?", bool(np.isnan(X).any()))
print("Feature order:", list(feature_names))
print("\nFirst 3 rows of X:")
print(X[:3])

X shape: (428, 3)
n (rows, months) = 428 | p (columns, features) = 3
y shape: (428,)
Any missing values in X? False
Feature order: ['CurrentRate', 'Trailing3MonthAverage', 'YearOverYearChange']

First 3 rows of X:
[[28.     28.      5.5356]
 [28.     28.      5.3756]
 [28.     28.      5.2406]]


In [4]:
# Transpose of the design matrix
Xt = X.T
print("X shape:  ", X.shape)
print("X^T shape:", Xt.shape)          # expected (3, 428)

print("\nFirst 3 columns of X^T (these are the first 3 months of X, written as columns):")
print(Xt[:, :3])

print("\n(X^T)^T equals X?", np.array_equal(Xt.T, X))

X shape:   (428, 3)
X^T shape: (3, 428)

First 3 columns of X^T (these are the first 3 months of X, written as columns):
[[28.     28.     28.    ]
 [28.     28.     28.    ]
 [ 5.5356  5.3756  5.2406]]

(X^T)^T equals X? True


In [5]:
# Compute the matrix product X^T X
XtX = X.T @ X
print("X^T X shape:", XtX.shape)
print(XtX)

# Quick sanity checks: XtX should be square and symmetric
print("\nSquare?   ", XtX.shape[0] == XtX.shape[1])
print("Symmetric?", np.allclose(XtX, XtX.T))

X^T X shape: (3, 3)
[[905123.0893 903474.4107  20914.344 ]
 [903474.4107 902003.8393  20449.495 ]
 [ 20914.344   20449.495    6212.9507]]

Square?    True
Symmetric? True


In [6]:
# Inspect one off-diagonal entry of X^T X (pick any i != j)
i, j = 0, 1   # CurrentRate (X1) and Trailing3MonthAverage (X2)
print(f"(X^T X)[{i},{j}] = {XtX[i, j]:.4f}  -- corresponds to features "
      f"'{feature_names[i]}' and '{feature_names[j]}'")

# Same entry computed by hand: dot product of column i and column j of X
manual = np.sum(X[:, i] * X[:, j])
print(f"Dot product of the two columns = {manual:.4f}")
print("Matches the matrix product?", np.isclose(manual, XtX[i, j]))

(X^T X)[0,1] = 903474.4107  -- corresponds to features 'CurrentRate' and 'Trailing3MonthAverage'
Dot product of the two columns = 903474.4107
Matches the matrix product? True


In [7]:
# Extra check for Q8: add one hypothetical new month and see how X and X^T X change.
x_new = X.mean(axis=0)                  # a made-up new row, just for the demo
X_new = np.vstack([X, x_new])           # X gets one more row
XtX_new = X_new.T @ X_new

print("X shape before:", X.shape, "-> after:", X_new.shape)
print("X^T X shape before:", XtX.shape, "-> after:", XtX_new.shape)

# X^T X changes by adding the outer product of the new row with itself
update = XtX + np.outer(x_new, x_new)
print("XtX_new equals XtX + x x^T ?", np.allclose(XtX_new, update))

X shape before: (428, 3) -> after: (429, 3)
X^T X shape before: (3, 3) -> after: (3, 3)
XtX_new equals XtX + x x^T ? True


### Answer these in your report

**Q5.** What are n and p for your dataset? What does each row of X represent? What does each column represent?

> *Your answer:* For our dataset, n = 428 and p = 3, so X is a 428 × 3 matrix (the code confirms this with `X.shape`). The 428 comes from the 441 calendar months between January 1990 and September 2026, minus the months lost when building the year-over-year feature (which needs a rate from 12 months earlier) and the next-month target (which needs a following month). Each row of X is one month, t, written as a snapshot of what was already known by the end of that month. Each column is one of the three predictors measured across all those months: the current monthly average rate (X1), the trailing three-month average (X2), and the year-over-year change (X3). The next-month rate that we want to predict is not inside X. It sits in the separate vector y, with one entry per row of X.

**Q6.** What are the dimensions of XᵀX, and why is that shape useful for the steps that come later?

> *Your answer:* XᵀX is 3 × 3. X is 428 × 3 and Xᵀ is 3 × 428, so the inner dimension of 428 is used up in the multiplication and only the feature count p is left on both sides. This is useful because every later step wants a small square matrix, and the number of months no longer matters. Only square matrices have determinants and can be checked for invertibility, and the normal equations for regression, (XᵀX)β = Xᵀy, need XᵀX to be invertible to solve for β. The same 3 × 3 shape also lines up with the covariance matrix we will build from the standardized features, whose eigenvalues and eigenvectors give the three principal components for PCA. The matrix is symmetric as well, which is the property that later lets us diagonalize it cleanly with real eigenvalues. So 428 rows of data get compressed into a 3 × 3 summary of how the features relate to each other.

**Q7.** Pick one single off-diagonal entry of XᵀX and explain, in words, what it measures about your two corresponding features.

> *Your answer:* We picked the entry in row 1, column 2 (index [0,1] in the code), which pairs the current rate (X1) with the trailing three-month average (X2). It is the dot product of those two columns, meaning we multiply the two feature values for each month and add up the results over all months. A large positive value says that X1 and X2 tend to be big in the same months, so they rise and fall together. Here we expect it to be very large, since both columns are peso amounts in the tens and X2 is built partly from X1 itself (it contains R_t as one of its three terms). It sits close to the diagonal entries (X1 with itself, and X2 with itself), which hints that the two features carry almost the same information and could cause multicollinearity. One caution is that the data is not centered yet, so this entry reflects the overall size of the rates as well as how they move together. It is not a correlation, and we should not read it as one until after standardizing.

**Q8.** If you added one new sample to your dataset, how would X change? Would XᵀX change in the same simple way? Explain.

> *Your answer:* Adding one more month would add one more row to X, so it would grow from 428 × 3 to 429 × 3. The old rows stay exactly as they were, and the new row holds that month's X1, X2, and X3. XᵀX would not grow, because it stays 3 × 3 no matter how many months we add. It would still change in a simple way, though: the new matrix equals the old XᵀX plus the outer product x xᵀ, where x is the new row written as a column vector. This is because XᵀX is the sum over all months of each row's outer product with itself, so one new month just contributes one more term to that sum. The cell above checks this numerically with `XtX_new` and `XtX + np.outer(x_new, x_new)`. The shape stays fixed while the entries get updated, so it is a small additive correction instead of a full rebuild.

### Save your work for the next notebook

In [8]:
%store X
%store XtX
print("Saved X, XtX for Part C.")

Stored 'X' (ndarray)
Stored 'XtX' (ndarray)
Saved X, XtX for Part C.
